In [1]:
import pandas as pd
import numpy as np
import re

# 1) Load
df = pd.read_csv("../data/raw/goodreadss_books.csv")

# 2) Treat empty/whitespace strings as NaN
df = df.replace(r"^\s*$", np.nan, regex=True)

# 3) Build 'year' if not present (extract from publish_date / first_publish_date)
def extract_year(val):
    if pd.isna(val):
        return np.nan
    m = re.search(r"(1[0-9]{3}|20[0-9]{2})", str(val))
    return m.group(0) if m else np.nan

if "year" not in df.columns:
    year = pd.Series(np.nan, index=df.index, dtype="object")
    if "publish_date" in df.columns:
        year = df["publish_date"].apply(extract_year)
    if "first_publish_date" in df.columns:
        year2 = df["first_publish_date"].apply(extract_year)
        year = year.fillna(year2)
    df["year"] = year

# 4) Required fields (dataset uses 'genres', not 'genre')
required_cols = ["genres", "author", "title", "year", "num_pages", "characters", "avg_rating"]
required_cols = [c for c in required_cols if c in df.columns]  # safety if a col is missing

# 5) Missing counts BEFORE cleaning
missing_before = df[required_cols].isna().sum().astype(int)

# 6) Drop rows ONLY if title is missing
dropped_due_to_title = int(df["title"].isna().sum()) if "title" in df.columns else 0
df_kept = df.dropna(subset=["title"]).copy() if "title" in df.columns else df.copy()

# 7) Fill other required fields with "Unknown"
fill_cols = [c for c in required_cols if c != "title"]
filled_counts = df_kept[fill_cols].isna().sum().astype(int)  # how many will be filled
df_kept[fill_cols] = df_kept[fill_cols].fillna("Unknown")

# 8) Summary table: how many removed/fixed per field
summary = pd.DataFrame({
    "field": required_cols,
    "missing_before": [missing_before[c] for c in required_cols],
    "rows_dropped": [dropped_due_to_title if c == "title" else 0 for c in required_cols],
    "rows_filled_with_Unknown": [0 if c == "title" else int(filled_counts.get(c, 0)) for c in required_cols],
})

print(summary.to_string(index=False))
print("\nRows original:", len(df))
print("Rows after title drop:", len(df_kept))

# 9) Save cleaned CSV
df_kept.to_csv("goodreads_cleaned.csv", index=False)
print("\nSaved: goodreads_cleaned.csv")


     field  missing_before  rows_dropped  rows_filled_with_Unknown
    genres             256             0                         0
    author             561             0                         2
     title             559           559                         0
      year             647             0                        88
 num_pages            1640             0                      1081
characters           15289             0                     14730
avg_rating             559             0                         0

Rows original: 20068
Rows after title drop: 19509

Saved: goodreads_cleaned.csv
